In [1]:
import re
import regex
assert "regex" in str(regex.compile("(test){e<=1}"))
import pandas as pd
import os
import numpy as np
pd.set_option('display.max_colwidth', 40)
import jiwer
import rapidfuzz
from jiwer import wer
from rapidfuzz.distance import Levenshtein
from pathlib import Path
project_dir = Path().resolve()
os.chdir(project_dir)

In [2]:
folder_path = "hand_typed_OCR_validation"

rows = []

for filename in os.listdir(folder_path):
    if filename.endswith(".txt"):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read() 
        rows.append({
            "filename": filename,
            "text": text
        })

df = pd.DataFrame(rows)

print(f'{len(df)} documents read')

31 documents read


## Extracting Rough Snippets

In [3]:
pattern1 = re.compile(
      r"Dear.*?(?:Sincer[a-z]{1,5}\b|Thank you for [Pp]art|Respetuosamente)",
      re.DOTALL)

pattern2 = re.compile(
      r"(?:\n|\A)\s*"
      r"Via[^\n]*\n"
      r"(?:\s*\n){0,3}"
      r"(?:\s*\S+(?:\s+\S+){0,5}\s*\n)?"
      r"(?:\s*\n){0,3}"
      r"(?:\s*\d{3,5}(?:\s+\d{3,5})+\s*\n)?\s*"
      r"(?:\s*\n){0,3}"
      r"(?:\s*\S+(?:\s+\S+){0,5}\s*\n)?"
      r"(?:\s*\n){0,3}"
      r"(?:\s*\S+(?:\s+\S+){0,5}\s*\n)?"
      r"(?:\s*\n){0,3}"
      r"(?:\s*\S+(?:\s+\S+){0,5}\s*\n)?"
      r"(?:\s*\n){0,3}"
      r"(?:[^\w\s]?Re[:;]\s*)?"
      r"(?:CPC\s*#?\s*|[^\w\s]#\s*)?"
      r"\d{2,3}-\d[\d!\])}]"
      r".*?"
      r"(?:Sincer[a-z]{1,5}\b|Thank you for [Pp]art|Respetuosamente)",
      re.DOTALL
  )

pattern3 = re.compile(
      r"(?:\n|\A)\s*"
      r"To Fi[lt]e\s*\n"                            
      r"(?:\s*\n){0,3}"
      r"(?:\s*\S+(?:\s+\S+){0,5}\s*\n)?"           
      r"(?:\s*\n){0,2}"
      r"(?:\s*\S+(?:\s+\S+){0,5}\s*\n)?"           
      r"(?:\s*\n){0,2}"
      r"(?:\s*\S+(?:\s+\S+){0,5}\s*\n)?"           
      r"(?:\s*\n){0,3}"
      r"(?:[^\w\s]?Re[:;]\s*)?"                     
      r"(?:CPC\s*#?\s*|[^\w\s]#\s*)?"              
      r"\d{2,3}-\d[\d!\])}]"                        
      r".*?"
      r"(?:Sincer[a-z]{1,5}\b|Thank you for [Pp]art|Respetuosamente)",
      re.DOTALL
  )

  # pattern4: catches 2018 and 2025 batch formats where the letter header is
  # just a date line followed directly by "Re: CPC# NNN-NN" — no Via, To File,
  # or Dear. Runs last so it never steals from pattern1/2/3.
pattern4 = re.compile(
      r"(?:\n|\A)\s*"
      r"(?:[^\w\s]?Re[:;]\s*)" 
      r"(?:CPC\s*#?\s*)?"
      r"\d{2,3}-\d[\d!\])}]"
      r".*?"
      r"(?:Sincer[a-z]{1,5}\b|Thank you for [Pp]art|Respetuosamente)",
      re.DOTALL
  )

pattern5 = re.compile(
    r"(?:\n|\A)\s*"
    r"Via[^\n]*\n"
    r"\n"
    r"(?:[^\n]*\n\s*\n){1,3}"
    r"(?:Re[:;]\s*)?"
    r"(?:CPC\s*#?\s*)?"
    r"\d{2,4}-\d{2}"
    r".*?"
    r"(?:Sincer[a-z]{1,5}\b|Thank you for [Pp]art|Respetuosamente)",
    re.DOTALL
)

In [4]:
WINDOW = 100

def add_window(text, start, end, window=100):
    """Add a window to the matched text."""
    start_ctx = max(0, start - window)
    end_ctx = min(len(text), end + window)
    return text[start_ctx:end_ctx]

def merge_spans(spans):
    if not spans:
        return []
    spans = sorted(spans)
    merged = [list(spans[0])]
    for s, e in spans[1:]:
        if s <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])
    return merged

def subtract_spans(text, spans):
    spans = merge_spans(spans)
    leftovers = []
    prev = 0
    for s, e in spans:
        if prev < s:
            leftovers.append(text[prev:s])
        prev = e
    if prev < len(text):
        leftovers.append(text[prev:])
    return leftovers

def overlaps(span, used_spans):
    s1, e1 = span
    for s2, e2 in used_spans:
        if not (e1 <= s2 or s1 >= e2):
            return True
    return False

rows = []

for _, row in df.iterrows():
    filename = row["filename"]
    text = row["text"]
    used_spans = []

    # PATTERN 1
    for m in pattern1.finditer(text):
        start, end = m.start(), m.end()
        if overlaps((start, end), used_spans):
            continue
        used_spans.append((start, end))
        rows.append({
            "filename": filename,
            "pattern": "pattern1",
            "snippet": add_window(text, start, end, WINDOW)
        })

    # PATTERN 2
    for m in pattern2.finditer(text):
        start, end = m.start(), m.end()
        if overlaps((start, end), used_spans):
            continue
        used_spans.append((start, end))
        rows.append({
            "filename": filename,
            "pattern": "pattern2",
            "snippet": add_window(text, start, end, WINDOW)
        })
        
    # PATTERN 3
    for m in pattern3.finditer(text):
        start, end = m.start(), m.end()
        if overlaps((start, end), used_spans):
            continue
        used_spans.append((start, end))
        rows.append({
            "filename": filename,
            "pattern": "pattern3",
            "snippet": add_window(text, start, end, WINDOW)
        })
        
    # PATTERN 4 
    for m in pattern4.finditer(text): 
        start, end = m.start(), m.end() 
        if overlaps((start, end), used_spans): 
            continue 
        used_spans.append((start, end)) 
        rows.append({ 
            "filename": filename, 
            "pattern": "pattern4", 
            "snippet": add_window(text, start, end, WINDOW) 
        })
    
    # PATTERN 5 
    leftover_chunks = subtract_spans(text, used_spans)
    final_unmatched_chunks = []
    for chunk in leftover_chunks:
        if "COMPLAINT" in chunk:
            chunk_used_spans = []
            for m in pattern5.finditer(chunk):
                start, end = m.start(), m.end()
                chunk_used_spans.append((start, end))
                rows.append({
                    "filename": filename,
                    "pattern": "pattern5",
                    "snippet": add_window(chunk, start, end, WINDOW)
                })
            chunk_leftovers = subtract_spans(chunk, chunk_used_spans)
            final_unmatched_chunks.extend(chunk_leftovers)
        else:
            final_unmatched_chunks.append(chunk)

    # UNMATCHED TEXT
    for chunk in final_unmatched_chunks:
        chunk = chunk.strip()
        if chunk:
            rows.append({
                "filename": filename,
                "pattern": "unmatched",
                "snippet": chunk
            })

df_combined = pd.DataFrame(rows)
df_combined["snippet"] = df_combined["snippet"].astype(str)

In [5]:
unmatched_text = df_combined.loc[df_combined["pattern"] == "unmatched"]
df_combined = df_combined.drop(unmatched_text.index)

## Capturing Case Numbers

In [6]:
df_combined["case_number"] = df_combined["snippet"].str.extract(r'CF?PC\s*#?4?\s*([0-9]+-[0-9]+)', flags=re.IGNORECASE)
df_combined["case_number_again"] = (                                                           
      df_combined["snippet"]                                                                     
      .str.extract(                                                                              
          r'(?:'
          r'[^\w\s]?Re[:;]\s*(?:CF?PC\s*#?4?\s*)?'                                               
          r'|CF?PC\s*#?4?\s*'                                                                    
          r'|[^\w\s]#\s*'                                                                      
          r')'                                                                                   
          r'([0-9]{2,3}-[0-9][\d!\])}])',                                                      
          flags=re.IGNORECASE                                                                    
      )[0]
      .str.replace(r'[\]!)}]$', '1', regex=True)                                                 
  )
# shelly_flag
df_combined["casenum_disagreement"] = np.where(
    (
        df_combined["case_number_again"].notna() &
        df_combined["case_number"].notna() &
        (df_combined["case_number"] != df_combined["case_number_again"])
    ),
    df_combined["case_number_again"],
    pd.NA
)


In [7]:
print(df_combined[df_combined['case_number'].isna() 
      & df_combined['case_number_again'].isna()])
print(df_combined[df_combined['casenum_disagreement'].notna()])

Empty DataFrame
Columns: [filename, pattern, snippet, case_number, case_number_again, casenum_disagreement]
Index: []
Empty DataFrame
Columns: [filename, pattern, snippet, case_number, case_number_again, casenum_disagreement]
Index: []


## Capturing Letter Date, Incident Date, Complaint Date

In [8]:
date_pattern = (
    r'(?:January|February|March|April|May|June|July|August|September|October|November|December)'
    r'\s+\d{1,2}'
    r'(?:,\s*\d{4})?')

DEAR = r"(?:Dear|Re:){e<=1}"

letter_date_pattern = (
    r'((?:January|February|March|April|May|June|July|August|September|October|November|December)'
    r'\s+\d{1,2},\s*\d{4})'
)

def extract_letter_date(text):
    if not isinstance(text, str):
        return None

    split = regex.search(
        r"^(.*?)(?:Dear|Re:)",
        text,
        flags=regex.IGNORECASE | regex.DOTALL
    )

    if split:
        m = regex.search(letter_date_pattern, split.group(1))
        if m:
            return m.group(0)

    lt = regex.search(
        r"(Letter\s*to){e<=1}.{0,50}",
        text,
        flags=regex.IGNORECASE | regex.DOTALL
    )

    if lt:
        m = regex.search(letter_date_pattern, lt.group(0))
        if m:
            return m.group(0)

    return None

df_combined["letter_date"] = df_combined["snippet"].apply(extract_letter_date)

incident_date_pattern = (
    r"occurred(?:\s+on)?(?:\s+or\s+about)?\s+"
    r"((?:January|February|March|April|May|June|July|August|September|October|November|December)"
    r"\s+\d{1,2},\s+\d{4})"
)

df_combined["incident_date"] = df_combined["snippet"].str.extract(
    incident_date_pattern,
    flags=re.I
)

In [9]:
percent_missing = df_combined.isnull().mean() * 100
print(percent_missing)

filename                  0.0
pattern                   0.0
snippet                   0.0
case_number               0.0
case_number_again         0.0
casenum_disagreement    100.0
letter_date               0.0
incident_date            77.5
dtype: float64


In [10]:
ed_1_complaint_date_pattern = re.compile(
    r'dear(?:\s+\S+){0,10}?\s+'
    r'('
    r'(?:January|February|March|April|May|June|July|August|September|October|November|December)'
    r'\s+\d{1,2}'
    r'(?:,\s*\d{4})?'
    r')',
    re.IGNORECASE
)
df_combined["complaint_date_1"] = df_combined["snippet"].str.extract(ed_1_complaint_date_pattern)

ed_1_complaint_date_pattern_2 = re.compile(
    rf'(?:'
    rf'({date_pattern})\s+against'
    rf'|'
    rf'against(?:\s+\S+){{0,10}}?\s+on\s+({date_pattern})'
    rf')',
    re.IGNORECASE
)

matches = df_combined["snippet"].str.extract(ed_1_complaint_date_pattern_2)
df_combined["complaint_date_2"] = matches[0].combine_first(matches[1])
df_combined["complaint_date"] = df_combined["complaint_date_1"].combine_first(df_combined["complaint_date_2"])

# shelly_flag
df_combined["disagreement"] = np.where(
    (
        df_combined["complaint_date_1"].notna() &
        df_combined["complaint_date_2"].notna() &
        (df_combined["complaint_date_1"] != df_combined["complaint_date_2"])
    ),
    df_combined["complaint_date_2"],
    pd.NaT
)

## Re-Indexing

In [11]:
df_combined["id_pre"] = df_combined.index

In [12]:
df_combined = df_combined.reset_index(drop=True)
df_combined["id_post"] = df_combined.index
df_combined = df_combined[["id_post"] + [c for c in df_combined.columns if c != "id_post"]]

In [13]:
converted = pd.to_datetime(df_combined["letter_date"], errors="coerce")

df_combined["letter_date"] = converted.where(
    converted.notna(),
    df_combined["letter_date"]
)
failed_parse_mask = df_combined["letter_date"].notna() & converted.isna()

failed_parses = df_combined.loc[
    failed_parse_mask,
    ["snippet", "letter_date"]
].copy()

failed_dates_df = df_combined.loc[
    converted.isna(), ].copy()
len(failed_dates_df)

0

## Splitting into Complaint, Investigation, Findings

In [14]:
DEAR = r"(?:Dear|Re:){e<=1}"
COMPLAINT = r"(?:COMPLAINT){e<=1}"
INVESTIGATION = r"(?:INVESTIGATION|INQUIRY){e<=1}"
CONCLUSION = r"(?:CONCLUSION){e<=1}"
FINDINGS = r"(?:[EF]INDINGS(?:\s+AND\s+CONCLUSIONS)?){e<=1}"
SINCERELY = r"(?:(?:Sincer[a-z]{1,5}){e<=1}|(?:Thank\syou))"
EVIDENCE = r"(?:EVIDENCE(?:[\s\.\-_:;]*)REVIEWED){e<=2}"
RIGHTS = r"(?:You\shave\sthe\sright\sto\appeal\s\this\sdecision){e<=3}"

CP_1_1 = regex.compile(
    rf"""
    {DEAR}
    (.*?)
    (?=
        {INVESTIGATION}
        |
        {FINDINGS}
        |
        {EVIDENCE}
        |
        {SINCERELY}
    )
    """,
    regex.DOTALL | regex.VERBOSE
)

CP_1_2 = regex.compile(
    rf"""
    {INVESTIGATION}
    (.*?)
    (?=
        {CONCLUSION}
        |
        {FINDINGS}
        |
        {SINCERELY}
    )
    """,
    regex.DOTALL | regex.VERBOSE
)

CP_1_3 = regex.compile(
    rf"""
    (?:
        {CONCLUSION}
        |
        {FINDINGS}
    )
    (.*?)
    (?={SINCERELY})
    """,
    regex.DOTALL | regex.VERBOSE
)

CP_2_1 = regex.compile(
    rf"""
    (?:
        {DEAR}
        |
        {COMPLAINT}
    )
    (.*?)
    (?=
        {EVIDENCE}
        |
        {INVESTIGATION}
    )
    """,
    regex.DOTALL | regex.VERBOSE
)

CP_2_2 = regex.compile(
    rf"""
    {EVIDENCE}
    (.*)
    """,
    regex.DOTALL | regex.IGNORECASE | regex.VERBOSE
)
        

def extract_group(text, pattern):
    if pd.isna(text):
        return pd.NA

    m = pattern.search(text)

    if not m:
        return pd.NA

    try:
        return m.group(1).strip()
    except Exception:
        return pd.NA


df_combined["complaint"] = df_combined["snippet"].apply(lambda x: extract_group(x, CP_1_1))
df_combined["investigation"] = df_combined["snippet"].apply(lambda x: extract_group(x, CP_1_2))
df_combined["findings"] = df_combined["snippet"].apply(lambda x: extract_group(x, CP_1_3))


later_pattern = regex.compile(r"(EVIDENCE[\s\.\-_:]*REVIEWED){e<=2}", regex.IGNORECASE)

later_pattern_mask = df_combined["snippet"].apply(
    lambda x: bool(later_pattern.search(str(x))))

df_combined.loc[later_pattern_mask, "complaint"] = df_combined.loc[later_pattern_mask, "snippet"].apply(
    lambda x: extract_group(x, CP_2_1))

df_combined.loc[later_pattern_mask, "findings"] = df_combined.loc[later_pattern_mask, "snippet"].apply(
    lambda x: extract_group(x, CP_2_2))

In [15]:
# Grabbing Complaint + Investigations and Putting into Complaint column, removing Investigations content.
fuzzy_end_pattern = regex.compile(
    r"(COMPLAINT AND){e<=2}\s*$",
    regex.IGNORECASE
)

mask_and = df_combined["complaint"].fillna("").apply(
    lambda x: bool(fuzzy_end_pattern.search(str(x).strip()))
)

df_combined.loc[mask_and, "complaint"] = (
    (
        df_combined.loc[mask_and, "complaint"]
        .fillna("")
        .str.strip()
        + " "
        + df_combined.loc[mask_and, "investigation"]
        .fillna("")
        .str.strip()
    )
    .str.strip()
)
df_combined.loc[mask_and, "investigation"] = pd.NA

In [16]:
df_combined.head()
print(len(df_combined))

40


In [17]:
percent_missing = df_combined.isnull().mean() * 100
print(percent_missing)
print(len(df_combined[df_combined["complaint"].isna()]))
print(df_combined[df_combined["complaint"].isna()])
print(len(df_combined))

id_post                   0.0
filename                  0.0
pattern                   0.0
snippet                   0.0
case_number               0.0
case_number_again         0.0
casenum_disagreement    100.0
letter_date               0.0
incident_date            77.5
complaint_date_1         80.0
complaint_date_2         80.0
complaint_date           65.0
disagreement            100.0
id_pre                    0.0
complaint                 0.0
investigation            70.0
findings                  5.0
dtype: float64
0
Empty DataFrame
Columns: [id_post, filename, pattern, snippet, case_number, case_number_again, casenum_disagreement, letter_date, incident_date, complaint_date_1, complaint_date_2, complaint_date, disagreement, id_pre, complaint, investigation, findings]
Index: []
40


## OCR Validation

In [18]:
df_gold = df_combined.copy()

In [19]:
df_ocr = pd.read_csv("df_ocr_samp.csv")

In [20]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

In [21]:
df_ocr["snippet"] = (
    df_ocr["complaint"].fillna("")
    + " "
    + df_ocr["investigation"].fillna("")
    + " "
    + df_ocr["findings"].fillna("")
).apply(normalize_text)

df_gold["snippet"] = (
    df_gold["complaint"].fillna("")
    + " "
    + df_gold["investigation"].fillna("")
    + " "
    + df_gold["findings"].fillna("")
).apply(normalize_text)


In [22]:
df_compare = (
    df_gold[["case_number", "snippet"]]
    .rename(columns={"snippet": "gold"})
    .merge(
        df_ocr[["case_number", "snippet"]]
        .rename(columns={"snippet": "ocr"}),
        on="case_number",
        how="inner"
    )
)

In [23]:
def cer(gold, ocr):
    if not gold:
        return None
    return Levenshtein.distance(gold, ocr) / len(gold)

df_compare["WER"] = df_compare.apply(
    lambda x: wer(x["gold"], x["ocr"]),
    axis=1
)

df_compare["CER"] = df_compare.apply(
    lambda x: cer(x["gold"], x["ocr"]),
    axis=1
)

In [24]:
print(f"Mean WER: {df_compare['WER'].mean():.2%}")
print(f"Median WER: {df_compare['WER'].median():.2%}")

Mean WER: 5.76%
Median WER: 4.47%


In [25]:
print(f"Mean CER: {df_compare['CER'].mean():.2%}")
print(f"Median CER: {df_compare['CER'].median():.2%}")

Mean CER: 3.28%
Median CER: 1.98%


In [26]:
gold_all = " ".join(df_compare["gold"].fillna(""))
orig_all  = " ".join(df_compare["ocr"].fillna(""))

total_wer = wer(gold_all, orig_all)
total_cer = (
    Levenshtein.distance(gold_all, orig_all)
    / len(gold_all)
)

print(f"Corpus WER: {total_wer:.3%}")
print(f"Corpus CER: {total_cer:.3%}")

Corpus WER: 5.609%
Corpus CER: 3.104%


In [27]:
total_chars = len(gold_all)
total_chars_no_space = len(gold_all.replace(" ", ""))
total_words = len(gold_all.split())

print(f"Total characters: {total_chars:,}")
print(f"Total characters (no spaces): {total_chars_no_space:,}")
print(f"Total words: {total_words:,}")

Total characters: 189,178
Total characters (no spaces): 158,655
Total words: 30,524
